# **TCN**

In [ ]:
import numpy as np
import pandas as pd
from preprocessing.preprocess import prep
from preprocessing.target import ttp_target
from metrics.Metrics import merged_metrics
name = "AFKS"
df: pd.DataFrame = pd.read_csv(f"/Users/side/Desktop/Trading Chaos AI/df/clean_df/{name}.csv")

TCN — это Temporal Convolutional Network, по-русски: сверточная сеть для временных рядов.
По сути это альтернатива RNN/LSTM/GRU, которая работает не через рекурсию, а через 1D-свёртки, но специально сконструированные под “только прошлое → будущее”.

In [ ]:

from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler

from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv1D, SpatialDropout1D, Add, ReLU, Dense, Lambda
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
df = pd.read_csv("Brent.csv")
df

,DateTime,Open,High,Low,Close,Alligator_Jaw,Alligator_Teeth,Alligator_Lips,Fractal_Up,Fractal_Down,...,AO_saucer_down,EntrySignal,EntryReason,Fractal_Up_conf,Fractal_Down_conf,AddOn_Anchor_Level,AddOn_Anchor_IsUp,AddOn_Size_Pct,AddOn_Ready,AddOn_Triggered
0,2015-10-26 10:00:00,48.05,48.12,47.89,48.09,48.28815,48.15758,48.06107,0,0,...,0,0,NaN,0,0,NaN,NaN,NaN,0,0
1,2015-10-26 11:00:00,48.10,48.36,48.00,48.30,48.26098,48.12601,48.05886,0,0,...,0,0,NaN,0,0,NaN,NaN,NaN,0,0
2,2015-10-26 12:00:00,48.30,48.35,48.18,48.30,48.23552,48.11588,48.05209,0,0,...,0,-1,three_color,0,0,NaN,NaN,NaN,0,0
3,2015-10-26 13:00:00,48.30,48.34,48.05,48.09,48.21971,48.10765,48.04267,0,0,...,0,0,NaN,0,0,NaN,NaN,NaN,0,0
4,2015-10-26 14:00:00,48.11,48.28,47.97,48.07,48.19550,48.09732,48.07014,0,0,...,0,0,NaN,0,0,NaN,NaN,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36239,2025-10-21 23:00:00,61.55,61.73,61.55,61.65,61.17612,61.14427,61.29423,0,0,...,0,0,NaN,0,0,61.52,1.0,0.3,0,0
36240,2025-10-22 09:00:00,62.27,62.66,62.26,62.46,61.16565,61.21749,61.31439,1,0,...,0,0,NaN,0,0,61.52,1.0,0.3,0,0
36241,2025-10-22 10:00:00,62.45,62.49,62.19,62.35,61.13752,61.23530,61.35151,0,0,...,0,0,NaN,0,0,61.52,1.0,0.3,0,0
36242,2025-10-22 11:00:00,62.35,62.55,62.25,62.36,61.14002,61.25526,61.40921,0,0,...,0,0,NaN,1,0,61.52,1.0,0.3,0,0


In [ ]:
scale_cols = [
    "Open", "High", "Low", "Close",
    "Alligator_Jaw", "Alligator_Teeth", "Alligator_Lips",
    "AO",
    "AddOn_Anchor_Level", "AddOn_Size_Pct"
]

scaler = RobustScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])


In [ ]:
columns = [
    "AddOn_Anchor_Level",
    "AddOn_Anchor_IsUp",
    "AddOn_Size_Pct"
]

df = df.dropna(subset=columns).reset_index(drop=True)

In [ ]:
import numpy as np
import pandas as pd

# если ещё не загружал:
# df = pd.read_csv("Brent.csv")

H = 20  # горизонт, можно менять

def add_goodtrade_target(df, horizon=20):
    df = df.copy()

    # будущая цена
    df['Close_fwd'] = df['Close'].shift(-horizon)

    # доходность по направлению сигнала
    ret_long = (df['Close_fwd'] - df['Close']) / df['Close']
    ret_short = (df['Close'] - df['Close_fwd']) / df['Close']

    ret = np.where(
        df['EntrySignal'] > 0, ret_long,
        np.where(df['EntrySignal'] < 0, ret_short, 0.0)
    )

    df['ret_H'] = ret

    # таргет
    df['GoodTrade'] = ((df['EntrySignal'] != 0) & (df['ret_H'] > 0)).astype(int)

    # убираем хвост, где нет Close_fwd
    df = df.iloc[:-horizon].reset_index(drop=True)
    return df

df = add_goodtrade_target(df, horizon=H)

print("'GoodTrade' в колонках:", 'GoodTrade' in df.columns)
print(df[['Close', 'Close_fwd', 'ret_H', 'EntrySignal', 'GoodTrade']].head())


'GoodTrade' в колонках: True
      Close  Close_fwd     ret_H  EntrySignal  GoodTrade
0 -0.847430  -0.879768  0.000000            0          0
1 -0.856136  -0.878939  0.000000            0          0
2 -0.859453  -0.876451  0.000000            0          0
3 -0.863599  -0.876036 -0.014402           -1          0
4 -0.870232  -0.878939  0.000000            0          0


In [ ]:
df

,DateTime,Open,High,Low,Close,Alligator_Jaw,Alligator_Teeth,Alligator_Lips,Fractal_Up,Fractal_Down,...,Fractal_Up_conf,Fractal_Down_conf,AddOn_Anchor_Level,AddOn_Anchor_IsUp,AddOn_Size_Pct,AddOn_Ready,AddOn_Triggered,Close_fwd,ret_H,GoodTrade
0,2015-10-26 19:00:00,-0.851161,-0.846918,-0.847880,-0.847430,-0.838401,-0.838317,-0.842386,1,0,...,0,1,-0.826423,0.0,0.0,1,1,-0.879768,0.000000,0
1,2015-10-26 20:00:00,-0.847015,-0.850641,-0.854530,-0.856136,-0.838041,-0.839319,-0.845333,0,0,...,0,0,-0.826423,0.0,0.0,0,0,-0.878939,0.000000,0
2,2015-10-26 21:00:00,-0.855721,-0.854365,-0.855362,-0.859453,-0.837932,-0.841183,-0.846775,0,0,...,1,0,-0.826423,0.0,0.0,0,0,-0.876451,0.000000,0
3,2015-10-26 22:00:00,-0.859038,-0.863881,-0.859102,-0.863599,-0.838056,-0.843231,-0.847305,0,0,...,0,0,-0.826423,0.0,0.0,0,0,-0.876036,-0.014402,0
4,2015-10-26 23:00:00,-0.863184,-0.867191,-0.864090,-0.870232,-0.838778,-0.844450,-0.848769,0,0,...,0,0,-0.826423,0.0,0.0,0,0,-0.878939,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36210,2025-10-20 18:00:00,-0.313433,-0.307406,-0.315461,-0.319237,-0.290522,-0.294306,-0.299641,0,0,...,0,1,-0.182927,0.0,0.0,0,0,-0.275705,0.000000,0
36211,2025-10-20 19:00:00,-0.315506,-0.317749,-0.310474,-0.313847,-0.291317,-0.295975,-0.303799,0,0,...,0,0,-0.182927,0.0,0.0,0,0,-0.242123,0.000000,0
36212,2025-10-20 20:00:00,-0.313433,-0.306578,-0.309227,-0.303483,-0.291634,-0.297384,-0.305587,0,0,...,0,0,-0.182927,0.0,0.0,0,0,-0.246683,0.000000,0
36213,2025-10-20 21:00:00,-0.303068,-0.303269,-0.298421,-0.304312,-0.292167,-0.300306,-0.306851,1,0,...,0,0,-0.182927,0.0,0.0,0,0,-0.246269,0.000000,0


In [ ]:
# предполагаем, что df уже есть и в нём посчитан GoodTrade

# сплит по времени: первые 80% баров — train, остальные — test
split_bar = int(len(df) * 0.8)

train_df = df.iloc[:split_bar].copy()
test_df  = df.iloc[split_bar:].copy()

print("Train bars:", len(train_df), "Test bars:", len(test_df))
print("Доля GoodTrade=1 train:", train_df['GoodTrade'].mean())
print("Доля GoodTrade=1 test :", test_df['GoodTrade'].mean())


Train bars: 28972 Test bars: 7243
Доля GoodTrade=1 train: 0.024126743062267017
Доля GoodTrade=1 test : 0.023470937456854895


In [ ]:
# только числовые колонки
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# что НЕ должно попасть в признаки (будущее/таргет)
drop_feature_cols = ['Close_fwd', 'ret_H', 'GoodTrade']

feature_cols = [c for c in numeric_cols if c not in drop_feature_cols]
print("Фичи для TCN:", feature_cols)

# сплит по времени: первые 80% баров — train, остальные — test
split_bar = int(len(df) * 0.8)
train_df = df.iloc[:split_bar].copy()
test_df  = df.iloc[split_bar:].copy()

print("Train bars:", len(train_df), "Test bars:", len(test_df))
print("Доля GoodTrade=1 train:", train_df['GoodTrade'].mean())
print("Доля GoodTrade=1 test :", test_df['GoodTrade'].mean())


Фичи для TCN: ['Open', 'High', 'Low', 'Close', 'Alligator_Jaw', 'Alligator_Teeth', 'Alligator_Lips', 'Fractal_Up', 'Fractal_Down', 'AO', 'Color AO', 'Alligator_Bullish', 'Alligator_Bearish', 'AlligatorStart_Long', 'AlligatorStart_Short', 'AO_sign', 'AO_zero_up', 'AO_zero_down', 'AO_three_green', 'AO_three_red', 'AO_saucer_up', 'AO_saucer_down', 'EntrySignal', 'Fractal_Up_conf', 'Fractal_Down_conf', 'AddOn_Anchor_Level', 'AddOn_Anchor_IsUp', 'AddOn_Size_Pct', 'AddOn_Ready', 'AddOn_Triggered']
Train bars: 28972 Test bars: 7243
Доля GoodTrade=1 train: 0.024126743062267017
Доля GoodTrade=1 test : 0.023470937456854895


In [ ]:
from sklearn.preprocessing import StandardScaler

# масштабируем признаки: fit на train, transform на train и test
scaler = StandardScaler()
train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
test_df[feature_cols]  = scaler.transform(test_df[feature_cols])

SEQ_LEN = 50  # длина окна истории, можно менять

def make_sequences(df_part, feature_cols, seq_len=50):
    data = df_part[feature_cols].values
    targets = df_part['GoodTrade'].values
    signals = df_part['EntrySignal'].values

    X_list, y_list = [], []

    for i in range(seq_len - 1, len(df_part)):
        # интересуют только бары с сигналом на последнем шаге окна
        if signals[i] == 0:
            continue

        X_seq = data[i - seq_len + 1 : i + 1, :]  # [seq_len, num_features]
        y_val = targets[i]

        X_list.append(X_seq)
        y_list.append(y_val)

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int32)
    return X, y

X_train, y_train = make_sequences(train_df, feature_cols, seq_len=SEQ_LEN)
X_test, y_test   = make_sequences(test_df,  feature_cols, seq_len=SEQ_LEN)

print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("Доля GoodTrade=1 в train seq:", (y_train == 1).mean())
print("Доля GoodTrade=1 в test seq :", (y_test == 1).mean())


X_train: (28923, 50, 30) X_test: (7194, 50, 30)
Доля GoodTrade=1 в train seq: 0.024133042907029008
Доля GoodTrade=1 в test seq : 0.023491798721156518


In [ ]:
def tcn_block(x, filters, kernel_size, dilation_rate, dropout=0.0):
    # первый conv
    conv1 = Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        padding='causal',
        dilation_rate=dilation_rate
    )(x)
    conv1 = ReLU()(conv1)
    conv1 = SpatialDropout1D(dropout)(conv1)

    # второй conv
    conv2 = Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        padding='causal',
        dilation_rate=dilation_rate
    )(conv1)
    conv2 = ReLU()(conv2)
    conv2 = SpatialDropout1D(dropout)(conv2)

    # residual-путь: приводим размер каналов, если нужно
    if x.shape[-1] != filters:
        x = Conv1D(filters, kernel_size=1, padding='same')(x)

    out = Add()([x, conv2])
    out = ReLU()(out)
    return out

time_steps = SEQ_LEN
num_features = X_train.shape[2]

inp = Input(shape=(time_steps, num_features))
x = inp

# набор дилатаций задаёт "глубину памяти"
for d in [1, 2, 4, 8, 16]:
    x = tcn_block(x, filters=64, kernel_size=3, dilation_rate=d, dropout=0.1)

# берём представление последнего шага по времени
x = Lambda(lambda t: t[:, -1, :])(x)

out = Dense(32, activation='relu')(x)
out = Dense(1, activation='sigmoid')(out)  # GoodTrade 0/1

model = Model(inputs=inp, outputs=out)
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 50, 30)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 50, 64)    │      5,824 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 50, 64)    │          0 │ conv1d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d   │ (None, 50, 64)    │          0 │ re_lu[0][0]       │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 50, 64)    │     12,352 │ spatial_dropout1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 50, 64)    │          0 │ conv1d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 50, 64)    │      1,984 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_1 │ (None, 50, 64)    │          0 │ re_lu_1[0][0]     │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 50, 64)    │          0 │ conv1d_2[0][0],   │
│                     │                   │            │ spatial_dropout1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 50, 64)    │          0 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 50, 64)    │     12,352 │ re_lu_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 50, 64)    │          0 │ conv1d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_2 │ (None, 50, 64)    │          0 │ re_lu_3[0][0]     │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 50, 64)    │     12,352 │ spatial_dropout1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_4 (ReLU)      │ (None, 50, 64)    │          0 │ conv1d_4[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_3 │ (None, 50, 64)    │          0 │ re_lu_4[0][0]     │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 50, 64)    │          0 │ re_lu_2[0][0],    │
│                     │                   │            │ spatial_dropout1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_5 (ReLU)      │ (None, 50, 64)    │          0 │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 50, 64)    │     12,352 │ re_lu_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_6 (ReLU)      │ (None, 50, 64)    │          0 │ conv1d_5[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_4 │ (None, 50, 64)    │          0 │ re_lu_6[0][0]     │
│ (SpatialDropout1D)  │                   │            │                 

 Total params: 121,089 (473.00 KB)

 Trainable params: 121,089 (473.00 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# веса классов из дисбаланса
pos = (y_train == 1).sum()
neg = (y_train == 0).sum()
scale_pos = neg / pos if pos > 0 else 1.0

class_weight = {0: 1.0, 1: scale_pos}
print("class_weight:", class_weight)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=64,
    callbacks=[early_stop],
    class_weight=class_weight,  # ← если хочешь без весов — просто убери этот параметр
    verbose=1
)


class_weight: {0: 1.0, 1: np.float64(40.43696275071633)}
Epoch 1/100
452/452 ━━━━━━━━━━━━━━━━━━━━ 68s 129ms/step - accuracy: 0.8138 - loss: 0.7823 - val_accuracy: 0.9725 - val_loss: 0.1645
Epoch 2/100
452/452 ━━━━━━━━━━━━━━━━━━━━ 82s 129ms/step - accuracy: 0.9727 - loss: 0.1768 - val_accuracy: 0.9746 - val_loss: 0.0997
Epoch 3/100
452/452 ━━━━━━━━━━━━━━━━━━━━ 81s 127ms/step - accuracy: 0.9699 - loss: 0.1739 - val_accuracy: 0.9746 - val_loss: 0.1072
Epoch 4/100
452/452 ━━━━━━━━━━━━━━━━━━━━ 81s 125ms/step - accuracy: 0.9733 - loss: 0.1283 - val_accuracy: 0.9746 - val_loss: 0.0893
Epoch 5/100
452/452 ━━━━━━━━━━━━━━━━━━━━ 57s 127ms/step - accuracy: 0.9738 - loss: 0.1234 - val_accuracy: 0.9744 - val_loss: 0.1032
Epoch 6/100
452/452 ━━━━━━━━━━━━━━━━━━━━ 55s 122ms/step - accuracy: 0.9732 - loss: 0.1354 - val_accuracy: 0.9746 - val_loss: 0.0821
Epoch 7/100
452/452 ━━━━━━━━━━━━━━━━━━━━ 58s 129ms/step - accuracy: 0.9729 - loss: 0.1261 - val_accuracy: 0.9746 - val_loss: 0.0885
Epoch 8/100
452/452

In [ ]:
y_proba = model.predict(X_test).ravel()
y_pred = (y_proba >= 0.5).astype(int)

print("TCN AUC:", roc_auc_score(y_test, y_proba))
print("\nОтчёт по классификации (TCN):")
print(classification_report(y_test, y_pred, digits=3))
print("Матрица ошибок (TCN):")
print(confusion_matrix(y_test, y_pred))


225/225 ━━━━━━━━━━━━━━━━━━━━ 9s 36ms/step
TCN AUC: 0.9867127124176126

Отчёт по классификации (TCN):
              precision    recall  f1-score   support

           0      1.000     0.974     0.987      7025
           1      0.480     1.000     0.649       169

    accuracy                          0.975      7194
   macro avg      0.740     0.987     0.818      7194
weighted avg      0.988     0.975     0.979      7194

Матрица ошибок (TCN):
[[6842  183]
 [   0  169]]
